# EPO Patent Search - Simplified Version

**Features:**
- Patent bibliographic information (title, abstract, applicants, inventors)
- CPC codes and classification
- Deep-tech domain classification (CPC to domain mapping)
- Strict fuzzy name validation (95% minimum match)
- Multi-sheet Excel output (Summary, Patent Details, CPC Reference, Deep-Tech Analysis)
- File-based caching (50-90% faster on repeated runs)
- Exponential backoff retry (automatic error recovery)
- Checkpoint system (resume capability)
- Modular architecture (maintainable code)
- Configuration management (easy parameter tuning)

**Note:** Patent family and legal status features have been removed from this version.

**Date:** 2025-11-28

## 1. Import Modules

In [ ]:
# Configuration and setup
from config.settings import load_config
from utils.logging_config import setup_logging
from utils.country_codes import COUNTRY_CODE_MAP
from utils.namespaces import BIBLIO_NAMESPACES

# Core functionality
from core.api_client import EPOClient

# Data I/O
from data.input_reader import read_company_names_from_excel
from data.output_writer import save_results_to_excel
from data.checkpoint_manager import CheckpointManager

# Processing
from processors.company_processor import clean_company_name_for_epo, search_and_process_company
from processors.deeptech_classifier import DeepTechClassifier  # NEW

# Standard library
import pandas as pd
from datetime import datetime
import time
import logging

print("✓ All modules imported successfully")

## 2. Load Configuration and Initialize

In [ ]:
# === INPUT FILE CONFIGURATION ===
# Set to filename to override config.json, or leave as None to use config default
INPUT_FILE = "2026 DT Startup List 3.xlsx"  # Change this as needed, or set to None

# Alternatively, you can specify full/relative paths:
# INPUT_FILE = "./data/my_companies.xlsx"
# INPUT_FILE = None  # Uses config.json default

# Load configuration
print("Loading configuration...")
config = load_config()

# Override input file if specified
if INPUT_FILE is not None:
    config['input']['excel_file'] = INPUT_FILE
    print(f"  📁 Input file: {INPUT_FILE} (override)")
else:
    print(f"  📁 Input file: {config['input']['excel_file']} (from config)")

# Setup logging
setup_logging(config)
logging.info("=" * 60)
logging.info("EPO Patent Search - Refactored Version")
logging.info("=" * 60)

# Initialize EPO API client
logging.info("Initializing EPO API client...")
epo_client = EPOClient(config)
logging.info("✓ EPO client initialized")

# Initialize checkpoint manager
logging.info("Initializing checkpoint manager...")
checkpoint_mgr = CheckpointManager(config['checkpoint']['checkpoint_dir'])
logging.info("✓ Checkpoint manager initialized")

# Initialize deep-tech classifier (NEW)
deeptech_classifier = None
if config.get('deeptech', {}).get('enabled', False):
    deeptech_settings = config['deeptech'].get('settings')
    if deeptech_settings:
        deeptech_classifier = DeepTechClassifier(
            deeptech_settings['mapping_file'],
            deeptech_settings
        )
        logging.info("✓ Deep-tech classifier initialized")
        print("  - Deep-tech classification: ENABLED")
    else:
        logging.warning("Deep-tech enabled but configuration failed to load")
        print("  - Deep-tech classification: DISABLED (config error)")
else:
    print("  - Deep-tech classification: DISABLED")

print("\n✓ Configuration loaded and systems initialized")
print(f"  - Cache enabled: {config['cache']['enabled']}")
print(f"  - Checkpoints enabled: {config['checkpoint']['enabled']}")
print(f"  - Max results per company: {config['api']['max_results_per_company']}")
print(f"  - Retry attempts: {config['retry']['max_attempts']}")

## 2.5. Configure Strict Validation Mode

This notebook uses **strict validation mode** to ensure only exact company name matches are flagged as "Validated".

**Strict Mode Features:**
- **Exact token matching**: Requires 95%+ of company name tokens to appear in the applicant name
- **Prevents false positives**: "Alfred Pte. Ltd." will NOT match inventor names like "Dr. Alfred Smith"
- **Configurable thresholds**: 
  - ≥95% = Validated (exact match)
  - 70-94% = Uncertain (partial match, review recommended)
  - <70% = Failed (no match)
- **All results included**: Even failed matches are kept in the output for manual review

**Usage:**
- Results are flagged but NOT excluded from the output
- Filter Excel by `validation_status = "Validated"` to see only exact matches
- Review "Uncertain" results manually to catch legitimate subsidiaries/variations

# === STRICT MATCHING CONFIGURATION OVERRIDE ===
# Override config to enable strict validation mode
print("\n⚙ Applying strict matching configuration overrides...")

config['search_validation'] = config.get('search_validation', {})
config['search_validation']['enabled'] = True
config['search_validation']['validation_mode'] = 'strict'  # NEW: Use strict token matching
config['search_validation']['min_similarity_score'] = 95   # Require 95% token overlap
config['search_validation']['uncertain_threshold'] = 70    # 70-94% = Uncertain
config['search_validation']['exclude_failed_validations'] = False  # Flag but don't exclude

# Optional: Test different CQL operators
config['search'] = config.get('search', {})
config['search']['cql_operator'] = 'pa'  # Change to 'pa_name' or 'apn' if testing applicant-only

# Disable aggressive suffix removal for strict searches (NEW)
config['name_normalization'] = config.get('name_normalization', {})
config['name_normalization']['remove_legal_suffixes'] = False  # Don't create "ALFRED" from "ALFRED PTE. LTD."

print("  ✓ Strict validation mode: ENABLED")
print(f"  ✓ Validation thresholds: {config['search_validation']['min_similarity_score']}% / {config['search_validation']['uncertain_threshold']}%")
print(f"  ✓ CQL operator: {config['search']['cql_operator']}")
print(f"  ✓ Exclude failed validations: {config['search_validation']['exclude_failed_validations']}")
print(f"  ✓ Legal suffix removal: DISABLED (strict search mode)")

## 3. Check for Existing Checkpoints (Resume Capability)

In [ ]:
# Check for existing checkpoints
summary_df = None
all_patent_details = []
start_index = 0

if checkpoint_mgr.is_checkpoint_available() and config['checkpoint']['auto_resume']:
    print("\n⚠ Existing checkpoint found!")
    
    # List available checkpoints
    checkpoints = checkpoint_mgr.list_checkpoints()
    if checkpoints:
        latest = checkpoints[-1]
        print(f"  Latest checkpoint: {latest['filename']}")
        print(f"  Run ID: {latest['metadata']['run_id']}")
        print(f"  Last processed: index {latest['metadata']['last_processed_index']}")
        print(f"  Checkpoint time: {latest['metadata']['checkpoint_timestamp']}")
    
    user_input = input("\nResume from checkpoint? (yes/no): ").strip().lower()
    
    if user_input in ['yes', 'y']:
        logging.info("Loading checkpoint...")
        checkpoint_data = checkpoint_mgr.load_checkpoint()
        
        if checkpoint_data:
            summary_df = checkpoint_data['summary_df']
            all_patent_details = checkpoint_data['details_list']
            start_index = checkpoint_data['last_index'] + 1
            
            print(f"\n✓ Checkpoint loaded successfully")
            print(f"  Resuming from index {start_index}")
            print(f"  Already processed: {len(all_patent_details)} patent details")
        else:
            print("\n✗ Failed to load checkpoint, starting fresh")
    else:
        print("\n→ Starting fresh (checkpoint ignored)")
else:
    print("\n→ No checkpoints found, starting fresh")

## 4. Load Input Data

In [ ]:
# Read company names from Excel
logging.info("Reading company names from Excel...")
companies = read_company_names_from_excel(
    config['input']['excel_file'],
    config['input']['sheet_name'],
    config['input']['column_name']
)

if not companies:
    print("\n✗ No company names found in input file. Exiting.")
    raise ValueError("No companies to process")

print(f"\n✓ Loaded {len(companies)} companies from input file")

# Initialize or verify summary DataFrame
if summary_df is None:
    logging.info("Initializing summary DataFrame...")
    summary_df = pd.DataFrame({
        'Original Name': companies,
        'Cleaned Name (for Query)': [clean_company_name_for_epo(name) for name in companies],
        'Status': 'Not Processed',
        'Search Error Message': pd.NA,
        'API Total Results': pd.NA,
        'Details Processed Count': 0,
        'Processed Country Distribution': pd.NA,
        'Processed Timestamp': pd.NaT
    })
    summary_df.index.name = 'Company Index'
    print("✓ Summary DataFrame initialized")
else:
    print("✓ Using existing summary DataFrame from checkpoint")

print(f"\nProcessing plan:")
print(f"  Total companies: {len(companies)}")
print(f"  Starting from index: {start_index}")
print(f"  Companies remaining: {len(companies) - start_index}")

## 5. Main Processing Loop

In [ ]:
# Main processing loop
processed_count = start_index
start_time = time.time()

logging.info("\nStarting main processing loop...")
logging.info("=" * 60)

for index, row in summary_df.iloc[start_index:].iterrows():
    original_name = row['Original Name']
    cleaned_name = row['Cleaned Name (for Query)']
    processed_count += 1
    
    logging.info(f"\n{'='*60}")
    logging.info(f"Processing {processed_count}/{len(summary_df)}: {original_name}")
    logging.info(f"{'='*60}")
    
    if not cleaned_name:
        logging.warning(f"Skipping processing for index {index} as cleaned name is empty")
        summary_df.loc[index, 'Status'] = 'Skipped - Empty Cleaned Name'
        summary_df.loc[index, 'Processed Timestamp'] = datetime.now()
        continue
    
    # Search and process company
    try:
        status, error_msg, api_count, details_list = search_and_process_company(
            original_name,
            cleaned_name,
            epo_client,
            BIBLIO_NAMESPACES,
            COUNTRY_CODE_MAP,
            config['api']['max_results_per_company'],
            config['api']['search_delay_seconds'],
            config['api']['detail_delay_seconds'],
            deeptech_classifier,
            config  # Pass config for validation and name normalization
        )
    except Exception as process_err:
        logging.error(f"!! UNEXPECTED ERROR processing company '{original_name}': {process_err}", exc_info=True)
        status = "Processing Error - Unexpected"
        error_msg = str(process_err)
        api_count = 'N/A'
        details_list = []
    
    # Update summary
    summary_df.loc[index, 'Status'] = status
    summary_df.loc[index, 'Search Error Message'] = error_msg if error_msg else pd.NA
    summary_df.loc[index, 'API Total Results'] = api_count
    summary_df.loc[index, 'Details Processed Count'] = len(details_list)
    summary_df.loc[index, 'Processed Timestamp'] = datetime.now()
    
    if details_list:
        all_patent_details.extend(details_list)
    
    # Checkpoint save
    if config['checkpoint']['enabled'] and processed_count % config['checkpoint']['checkpoint_interval'] == 0:
        logging.info(f"\nSaving checkpoint at index {index}...")
        try:
            checkpoint_mgr.save_checkpoint(
                summary_df,
                all_patent_details,
                index,
                len(summary_df),
                statistics={
                    'processed_count': processed_count,
                    'total_details': len(all_patent_details),
                    'cache_stats': epo_client.get_cache_stats()
                }
            )
        except Exception as checkpoint_err:
            logging.error(f"Failed to save checkpoint: {checkpoint_err}")
    
    # Delay between companies
    if processed_count < len(summary_df):
        logging.debug(f"\n(Waiting {config['api']['search_delay_seconds']}s before next company search...)")
        time.sleep(config['api']['search_delay_seconds'])

end_time = time.time()
logging.info("\n" + "="*60)
logging.info("Main processing loop completed")
logging.info("="*60)

print(f"\n✓ Processing completed in {end_time - start_time:.2f} seconds")

## 6. Post-Processing: Calculate Country Distribution

In [ ]:
# Calculate unique publication country distribution
logging.info("Calculating country distribution...")

if all_patent_details:
    details_df_for_stats = pd.DataFrame(all_patent_details)
    
    if 'country_code' in details_df_for_stats.columns and 'original_search_name' in details_df_for_stats.columns:
        country_distribution = details_df_for_stats.groupby('original_search_name')['country_code'].apply(
            lambda x: ", ".join(sorted(list(x.dropna().astype(str).unique())))
        )
        summary_df['Processed Country Distribution'] = summary_df['Original Name'].map(country_distribution).fillna("")
        logging.info("✓ Country distribution calculated")
    else:
        missing_cols = [col for col in ['country_code', 'original_search_name'] if col not in details_df_for_stats.columns]
        logging.warning(f"Required column(s) {missing_cols} not found in detailed data. Cannot calculate distribution.")
        summary_df['Processed Country Distribution'] = "Error: column(s) missing"
else:
    logging.info("No detailed patent data collected, skipping country distribution calculation.")
    summary_df['Processed Country Distribution'] = ""

print("✓ Post-processing complete")

## 7. Save Results to Excel

In [ ]:
import importlib
from data import output_writer
importlib.reload(output_writer)

# Generate output filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_filename = f"{config['output']['filename_prefix']}{timestamp}.xlsx"
output_path = f"{config['output']['output_dir']}/{output_filename}"

# Save results
logging.info(f"\nSaving results to {output_path}...")
save_results_to_excel(summary_df, all_patent_details, output_path)

# Clean up checkpoints after successful completion
if config['checkpoint']['enabled']:
    logging.info("Cleaning up checkpoints...")
    checkpoint_mgr.cleanup()

print(f"\n✓ Results saved to: {output_filename}")

## 8. Display Summary Statistics

In [ ]:
# Display final summary
print("\n" + "="*60)
print("PROCESSING COMPLETE")
print("="*60)
print(f"\nTotal time: {end_time - start_time:.2f} seconds")

print("\n--- Status Summary ---")
status_counts = summary_df['Status'].value_counts()
print(status_counts.to_string())

print("\n--- Cache Statistics ---")
cache_stats = epo_client.get_cache_stats()
for key, value in cache_stats.items():
    print(f"  {key}: {value}")

print(f"\n--- Output ---")
print(f"  File: {output_filename}")
print(f"  Location: {config['output']['output_dir']}")
print(f"  Total patents processed: {len(all_patent_details)}")

print("\n" + "="*60)
logging.info("Script finished successfully")
print("✓ Script finished successfully")